In [ ]:
import geopandas as gpd
import gpxpy
import numpy as np
import pandas as pd
from haversine import Unit, haversine_vector
import os
import plotly.express as px
from dem_stitcher import stitch_dem
import elevation
import rasterio
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy import ndimage
import math

In [ ]:
gpx_store = "/home/wcn/SynologyDrive/19. GPS/19.10_Hikes/AOT/2026/"
gpx_file = "20260523_AOT_K1-S1.gpx"
gpx_filepath = os.path.join(gpx_store, gpx_file)

dem_folder = "/home/wcn/.map_cache/dem"

In [ ]:
def roundup(x, nearest_value):
    return int(math.ceil(x / nearest_value)) * nearest_value


def rounddown(x, nearest_value):
    return int(math.floor(x / nearest_value)) * nearest_value

In [ ]:
with open(gpx_filepath, "r") as gpx_content:
    gpx = gpxpy.parse(gpx_content)

data = []

for track in gpx.tracks:
    for segment in track.segments:
        for point in segment.points:
            data.append(
                {
                    "lat": point.latitude,
                    "lon": point.longitude,
                    "elev": point.elevation,
                    "time": point.time,
                }
            )

df = pd.DataFrame(data)

In [ ]:
df["time"] = df["time"].dt.tz_convert("UTC")

coords = df[["lat", "lon"]].to_numpy()

distances_segment = np.zeros(len(df))
distances_segment[1:] = haversine_vector(coords[:-1], coords[1:], unit=Unit.METERS)

df["distance"] = np.cumsum(distances_segment)

df["duration"] = df["time"].diff().fillna(0)


gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326"
)


In [ ]:
tmp = gpd.read_file(gpx_filepath, layer="track_points")
tmp.info()

In [ ]:
minx, miny, maxx, maxy = gdf.dissolve().bounds.loc[0]

minx, miny, maxx, maxy = bounds = (
    rounddown(minx, 0.05) - 0.05,
    rounddown(miny, 0.05) - 0.05,
    roundup(maxx, 0.05) + 0.05,
    roundup(maxy, 0.05) + 0.05,
)

In [ ]:
elevation.clip(bounds=bounds, output=os.path.join(dem_folder, "dem_elevation.tif"))
elevation.clean()

In [ ]:
dem_data = rasterio.open(os.path.join(dem_folder, "dem_elevation.tif"))
dem_array = dem_data.read(1)

In [ ]:
dem_array_float = dem_array.astype(float)

dem_array_float = ndimage.gaussian_filter(dem_array_float, sigma=0.33)

upscale_factor = 5

dem_array_float = ndimage.zoom(dem_array_float, upscale_factor).round(decimals=1)

transform = dem_data.transform * dem_data.transform.scale(
    (dem_data.width / dem_array_float.shape[-1]),
    (dem_data.height / dem_array_float.shape[-2]),
)

In [ ]:
df["elevation"] = [
    dem_array[dem_data.index(x, y)] for x, y in zip(df["lon"], df["lat"], strict=True)
]
df["elevation"] = df["elevation"].astype(float)

gdf["elevation"] = [
    dem_array[dem_data.index(x, y)]
    for x, y in zip(gdf["geometry"].x, gdf["geometry"].y, strict=True)
]
gdf["elevation"] = gdf["elevation"].astype(float)

gdf["elevation_resample"] = dem_array_float[
    rasterio.transform.rowcol(transform, gdf["geometry"].x, gdf["geometry"].y)
]
df["elevation_resample"] = dem_array_float[
    rasterio.transform.rowcol(transform, gdf["geometry"].x, gdf["geometry"].y)
]

In [ ]:
X, p = stitch_dem(
    bounds,
    dem_name="glo_30",  # Global Copernicus 30 meter resolution DEM
    dst_ellipsoidal_height=False,
    dst_area_or_point="Point",
)
with rasterio.open(os.path.join(dem_folder, "dem.tif"), "w", **p) as ds:
    ds.write(X, 1)
    ds.update_tags(AREA_OR_POINT="Point")

In [ ]:
dem_data = rasterio.open(os.path.join(dem_folder, "dem.tif"))
dem_array = dem_data.read(1)

dem_array_float = dem_array.astype(float)

dem_array_float = ndimage.gaussian_filter(dem_array_float, sigma=0.75)

upscale_factor = 5

dem_array_float = ndimage.zoom(dem_array_float, upscale_factor).round(decimals=1)

transform = dem_data.transform * dem_data.transform.scale(
    (dem_data.width / dem_array_float.shape[-1]),
    (dem_data.height / dem_array_float.shape[-2]),
)

In [ ]:
gdf["ele"] = [
    dem_array[dem_data.index(x, y)]
    for x, y in zip(gdf["geometry"].x, gdf["geometry"].y, strict=True)
]

gdf["ele"] = gdf["ele"].astype(float)
df["ele"] = gdf["ele"]

gdf["ele_resample"] = dem_array_float[
    rasterio.transform.rowcol(transform, gdf["geometry"].x, gdf["geometry"].y)
]
df["ele_resample"] = gdf["ele_resample"]

gdf.head()


In [ ]:
fig_1 = px.scatter(df, x="lon", y="lat", template="plotly_dark")

fig_1.show()


In [ ]:
fig_2 = px.line(
    df,
    x="time",
    y=["elev", "ele", "ele_resample", "elevation", "elevation_resample"],
    template="plotly_dark",
)

fig_2.show()


In [ ]:
fig = go.Figure(
    data=[
        go.Scatter3d(
            x=df["lon"],
            y=df["lat"],
            z=df["elevation"],
            mode="markers",
            marker=dict(
                size=5, color=df["elevation"], colorscale="portland", opacity=0.8
            ),
        )
    ]
)

# tight layout
fig.update_layout(margin=dict(l=0, r=0, b=0, t=0), template="plotly_dark")
fig.show()

In [ ]:
fig_4 = px.line_map(
    df,
    lat="lat",
    lon="lon",
    hover_name="time",
    zoom=11,
    map_style="open-street-map",
)

fig_4.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0})

fig_4.show()


In [ ]:
vmin = rounddown(dem_array.min(), 100)
vmax = roundup(dem_array.max(), 100)
extent = minx, maxx, miny, maxy

fig, ax = plt.subplots()
cax = plt.imshow(dem_array, extent=extent, cmap="terrain", vmin=vmin, vmax=vmax)
fig.colorbar(cax, ax=ax)
gdf.plot(ax=ax, markersize=1, zorder=11, color="red")

ax.set(xlim=[minx, maxx], ylim=[miny, maxy])

plt.show()

In [ ]:
sns.displot(dem_array.ravel())